In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Model sizes in parameters (approximate)
size_to_params = {
    "20M": 20e6,
    "59M": 59e6,
    "136M": 136e6,
    "267M": 267e6
}

# Seeds
seeds = [132, 479, 865]

# Define compute estimation function
def compute_log_flops(size, tokens):
    params = size_to_params[size]
    return np.log10(6 * params * tokens)

# File setup
checkpoint_template = "OLMo_{size}_{seed}-{hash_str}/latest_unsharded"
file_name = "ppl-validation"

# Plot for each seed
for seed in seeds:
    plt.figure(figsize=(8, 6))

    all_logs = []
    all_avg_logs = []

    for size in size_to_params.keys():
        # Load losses
        checkpoint_dir = checkpoint_template.format(size=size, seed=seed, hash_str="*")
        losses = torch.load(f"{checkpoint_dir}/{file_name}_losses.pt")  # N x (T-1)

        # Get token losses (last column)
        token_losses = losses[:, -1].numpy()
        
        # Compute log compute
        log_compute = compute_log_flops(size, 5.4e9)  # Adding 1 to match full sequence length

        # Store for global averaging
        all_logs.append((log_compute, token_losses))
        all_avg_logs.append((log_compute, token_losses.mean()))

        # Plot individual losses as light lines
        for loss in token_losses:
            plt.plot(log_compute, loss, 'c', alpha=0.1)  # Light cyan lines
        
    # Plot averaged losses as a thick line
    log_computes, avg_losses = zip(*all_avg_logs)
    plt.plot(log_computes, avg_losses, 'b-', linewidth=2, label=f"Avg Loss (Seed {seed})")

    plt.xlabel("log Compute (log FLOPs)")
    plt.ylabel("Token Loss")
    plt.title(f"Token-Level Scaling Laws (Seed {seed})")
    plt.legend()
    plt.grid(True)

    plt.show()


ImportError: libffi.so.6: cannot open shared object file: No such file or directory